In [ ]:
from photutils import background
from photutils import psf

In [ ]:
import sys
sys.path.append("..")
sys.path.append("../../imaging/")

In [ ]:
from phot_utils import show_image

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
from astropy import units as u

In [ ]:
import numpy as np

In [ ]:
import astropy
from astropy.nddata import CCDData
from astropy.table import Table

In [ ]:
import forced_photometry 

In [ ]:
obj = "yasone1"
filt = "r"
img_name = "01"

In [ ]:
cd ..

In [ ]:
img = forced_photometry.load_image(obj, filt, img_name)

In [ ]:
img_coadd = CCDData.read(f"{obj}/coadd_median_{filt}/coadd.fits", unit="adu")

In [ ]:
cat = Table.read(f"{obj}/coadd_median_{filt}/detection.cat", hdu=2)

In [ ]:
cat_frame = Table.read(f"{obj}/img_{filt}_{img_name}/detection.cat", hdu=2)

In [ ]:
cat_coords = img_coadd.wcs.pixel_to_world(cat["X_IMAGE"]-1, cat["Y_IMAGE"]-1)

In [ ]:
x_cat_all, y_cat_all = img.wcs.world_to_pixel(cat_coords)

In [ ]:
psf_size = 25

In [ ]:
filt_inbounds = x_cat_all > 0 + psf_size/2
filt_inbounds &= x_cat_all < img.shape[1] - psf_size/2 - 1
filt_inbounds &= y_cat_all > 0 + psf_size/2
filt_inbounds &= y_cat_all < img.shape[0] - psf_size/2 - 1
filt_inbounds &= cat["FLAGS"] < 64
filt_inbounds &= cat["IMAFLAGS_ISO"] < 4

In [ ]:
x_cat = x_cat_all[filt_inbounds]
y_cat = y_cat_all[filt_inbounds]

In [ ]:
bkg_estimator = background.SExtractorBackground()

In [ ]:
img.shape

In [ ]:
2000 / 16

In [ ]:
bkg = background.Background2D(img, (256, 256), filter_size=(1, 1),
                              bkg_estimator=bkg_estimator)

In [ ]:
bkg.background_rms_median

In [ ]:
show_image(bkg.background.data, log=True)

In [ ]:
img_nobkg = CCDData(img- bkg.background, unit="adu")

In [ ]:
show_image(img_nobkg, log=True)

In [ ]:
plt.figure(dpi = 600)
show_image(img_nobkg.data, log=True)
plt.scatter(x_cat, y_cat, s=5, lw=0, color="C1", alpha=0.3)

In [ ]:
bkg.background_rms_median.value

In [ ]:
psf_size = 25

hsize = (psf_size - 1) / 2
x = cat_frame['X_IMAGE'] - 1
y = cat_frame['Y_IMAGE'] - 1
mask_edge = ((x > hsize) & (x < (img.shape[1] - 1 - hsize)) &
        (y > hsize) & (y < (img.shape[0] - 1 - hsize)))


In [ ]:
filt_good = cat_frame["FLAGS"] == 0
filt_good &= cat_frame["FLAGS_WEIGHT"] == 0
filt_good &= cat_frame["FWHM_IMAGE"] < 6
filt_good &= cat_frame["ELLIPTICITY"] < 0.3

filt_good &= cat_frame["MAGERR_APER"][:, 3] < 0.01
filt_good &=  mask_edge

In [ ]:
good_stars_tbl = Table()
good_stars_tbl["x"] = x[filt_good]
good_stars_tbl["y"] = y[filt_good]

In [ ]:
good_stars_tbl

In [ ]:
stars = psf.extract_stars(img_nobkg, good_stars_tbl, size=psf_size)


In [ ]:
from astropy.visualization import simple_norm
nrows = 10
ncols = 5
fig, ax = plt.subplots(nrows=nrows, ncols=ncols, figsize=(20, 20),
                       squeeze=True)
ax = ax.ravel()
for i in range(nrows * ncols):
    norm = simple_norm(stars[i], 'log', percent=99.0)
    ax[i].imshow(stars[i], norm=norm, origin='lower', cmap='viridis')


In [ ]:
epsf_builder = psf.EPSFBuilder(oversampling=3, maxiters=5,
                           progress_bar=True)
epsf, fitted_stars = epsf_builder(stars)


In [ ]:
show_image(epsf.data, log=True, figsize=(5, 5), dpi=100, clim=(0, None))


In [ ]:
psf_model = psf.MoffatPSF()
psf_model.alpha.bounds = [0, 30] # radius
psf_model.beta.bounds = [0.5, 6] # power
psf_model.alpha.fixed = False
psf_model.beta.fixed = False

In [ ]:
shape = epsf.data.shape
psfphot = psf.PSFPhotometry(psf_model, shape, aperture_radius=5,  xy_bounds=5)
init_params = Table()
init_params["x"] = [shape[0]/2]
init_params["y"] = [shape[1]/2]

result = psfphot(epsf.data, init_params=init_params) 

In [ ]:
result

In [ ]:
psf_model_fit = psf.MoffatPSF(alpha = result["alpha_fit"][0]/3, beta=result["beta_fit"][0])

In [ ]:
show_image(psfphot.make_model_image(shape), dpi=100, log=True, clim=(None, None))

In [ ]:
show_image(psfphot.make_residual_image(epsf.data) / epsf.data - 1, dpi=100, cmap="RdBu", clim=(-2, 2))

In [ ]:
show_image(psfphot.make_residual_image(epsf.data), dpi=100, cmap="RdBu")

In [ ]:
psf_size = 25

In [ ]:
psfphot = psf.PSFPhotometry(epsf, (psf_size, psf_size), aperture_radius=5,  xy_bounds=5)


In [ ]:
psf_ana = psf_model_fit

In [ ]:
psf_ana.beta

In [ ]:
psf_ana.fixed["alpha"] = True
psf_ana.fixed["beta"] = True

In [ ]:
init_params = Table()
init_params["x"] = x_cat
init_params["y"] = y_cat

In [ ]:
phot = psfphot(img_nobkg, init_params = init_params) 

In [ ]:
psfphot_ana = psf.PSFPhotometry(psf_ana, (psf_size, psf_size), aperture_radius=5,  xy_bounds=5)


In [ ]:
phot_ana = psfphot_ana(img_nobkg, init_params = init_params, )

In [ ]:
phot

In [ ]:
phot_ana

In [ ]:
show_image(img_nobkg, log=True)

In [ ]:
show_image(img_nobkg, log=True)
plt.scatter(phot["x_fit"], phot["y_fit"], lw=0.5, s=10, color="none", ec="C1")

In [ ]:
img_model = psfphot.make_model_image(img.shape)
show_image(img_model.data, log=True)

In [ ]:
img_model = psfphot_ana.make_model_image(img.shape)
show_image(img_model.data, log=True)

In [ ]:
img_resid = psfphot_ana.make_residual_image(img_nobkg)

In [ ]:
show_image(img_resid, log=True, clim=(2000, 0))

In [ ]:
img_resid = psfphot.make_residual_image(img_nobkg)

In [ ]:
show_image(img_resid, log=True, clim=(2000, 0))

In [ ]:
plt.hist(phot["x_fit"] - phot["x_init"], 100)

In [ ]:
plt.hist(phot["y_fit"] - phot["y_init"], 100)